# Trabalho 3 - CG

### Imports

In [401]:
# !pip install pyopengl glfw pyglm numpy pillow

In [402]:
import glfw
from OpenGL.GL import *
import numpy as np
import glm
import math
import os
from numpy import random
from PIL import Image
import json

from shader_s import Shader

### Inicializando janela

In [403]:
glfw.init()
glfw.window_hint(glfw.VISIBLE, glfw.FALSE)

altura = 700
largura = 700

window = glfw.create_window(largura, altura, "Programa", None, None)

if (window == None):
    print("Failed to create GLFW window")
    glfw.terminate()
    
glfw.make_context_current(window)



(python:5903): Gtk-WARNING **: 20:21:55.610: gtk_disable_setlocale() must be called before gtk_init()


### Shaders

In [404]:
mainShader = Shader("vertex_shader.vs", "fragment_shader.fs")
skyboxShader = Shader("skybox.vs", "skybox.fs")

program = mainShader.getProgram()
skyboxProgram = skyboxShader.getProgram()

### Carregando Modelos

In [405]:
glEnable(GL_TEXTURE_2D)
glHint(GL_LINE_SMOOTH_HINT, GL_DONT_CARE)
glEnable( GL_BLEND )
glBlendFunc( GL_SRC_ALPHA, GL_ONE_MINUS_SRC_ALPHA )
glEnable(GL_LINE_SMOOTH)


global vertices_list
vertices_list = []    
global textures_coord_list
textures_coord_list = []
global normals_list
normals_list = [] 


def load_model_from_file(filename):
    """Loads a Wavefront OBJ file. """
    objects = {}
    vertices = []
    texture_coords = []
    faces = []

    material = None

    # abre o arquivo obj para leitura
    for line in open(filename, "r"): ## para cada linha do arquivo .obj
        if line.startswith('#'): continue ## ignora comentarios
        values = line.split() # quebra a linha por espaço
        if not values: continue

        ### recuperando vertices
        if values[0] == 'v':
            vertices.append(values[1:4])

        ### recuperando coordenadas de textura
        elif values[0] == 'vt':
            texture_coords.append(values[1:3])

        ### recuperando faces 
        elif values[0] in ('usemtl', 'usemat'):
            material = values[1]
        elif values[0] == 'f':
            face = []
            face_texture = []
            for v in values[1:]:
                w = v.split('/')
                face.append(int(w[0]))
                if len(w) >= 2 and len(w[1]) > 0:
                    face_texture.append(int(w[1]))
                else:
                    face_texture.append(0)

            faces.append((face, face_texture, material))

    model = {}
    model['vertices'] = vertices
    model['texture'] = texture_coords
    model['faces'] = faces

    return model

def load_texture_from_file(img_textura):
    texture_id = glGenTextures(1) # Pede um id a opengl
    
    glBindTexture(GL_TEXTURE_2D, texture_id)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_S, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_WRAP_T, GL_REPEAT)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_2D, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    
    img = Image.open(img_textura)
    img = img.convert('RGBA') # Converte pra RGBA
    
    img_width = img.size[0]
    img_height = img.size[1]
    image_data = img.tobytes("raw", "RGBA", 0, -1)
    
    glTexImage2D(GL_TEXTURE_2D, 0, GL_RGBA, img_width, img_height, 0, GL_RGBA, GL_UNSIGNED_BYTE, image_data)
    
    return texture_id # Retorna o ID


'''
É possível encontrar, na Internet, modelos .obj cujas faces não sejam triângulos. Nesses casos, precisamos gerar triângulos a partir dos vértices da face.
A função abaixo retorna a sequência de vértices que permite isso. Créditos: Hélio Nogueira Cardoso e Danielle Modesti (SCC0650 - 2024/2).
'''
def circular_sliding_window_of_three(arr):
    if len(arr) == 3:
        return arr
    circular_arr = arr + [arr[0]]
    result = []
    for i in range(len(circular_arr) - 2):
        result.extend(circular_arr[i:i+3])
    return result
    
global numberTextures
numberTextures = 0

def load_obj_and_texture(objFile, texturesList):
    modelo = load_model_from_file(objFile)
    
    ### inserindo vertices do modelo no vetor de vertices
    verticeInicial = len(vertices_list)
    print('Processando modelo {}. Vertice inicial: {}'.format(objFile, len(vertices_list)))
    faces_visited = []
    for face in modelo['faces']:
        if face[2] not in faces_visited:
            faces_visited.append(face[2])
        for vertice_id in circular_sliding_window_of_three(face[0]):
            vertices_list.append(modelo['vertices'][vertice_id - 1])
        for texture_id in circular_sliding_window_of_three(face[1]):
            textures_coord_list.append(modelo['texture'][texture_id - 1])
        
    verticeFinal = len(vertices_list)
    print('Processando modelo {}. Vertice final: {}'.format(objFile, len(vertices_list)))
    
    tid = load_texture_from_file(texturesList[0])
    
    return verticeInicial, verticeFinal - verticeInicial, tid

In [406]:
def load_mtl(mtl_path):
    """
    Lê um arquivo .mtl e retorna um dicionário de materiais.
    Formato retornado: {nome_material: {'map_Kd': caminho_textura_ou_None, 'Kd': [r, g, b]}}
    """
    materials = {}
    current = None
    mtl_dir = os.path.dirname(os.path.abspath(mtl_path))

    for line in open(mtl_path, 'r', encoding='utf-8', errors='ignore'):
        if line.startswith('#'):
            continue
        values = line.split()
        if not values:
            continue
        if values[0] == 'newmtl':
            current = values[1]
            materials[current] = {'map_Kd': None, 'Kd': [1.0, 1.0, 1.0]}
        elif values[0] == 'Kd' and current:
            materials[current]['Kd'] = [float(v) for v in values[1:4]]
        elif values[0] == 'map_Kd' and current:
            # suporta caminhos com espaços e separadores diferentes de SO
            tex_rel = ' '.join(values[1:]).strip().replace('\\', os.sep).replace('/', os.sep)
            materials[current]['map_Kd'] = os.path.join(mtl_dir, tex_rel)

    return materials

def load_obj_with_mtl(obj_path, texture_overrides=None, uv_mult_u=1.0, uv_mult_v=1.0):
    global vertices_list, textures_coord_list, normals_list

    obj_dir = os.path.dirname(os.path.abspath(obj_path))

    # ── 1. Parse do arquivo OBJ ──────────────────────────────────────────────
    raw_vertices   = []
    raw_tex_coords = []
    raw_normals    = [] 
    
    faces_por_material = {}   # {nome_material: [(face_verts, face_uvs, face_vns), ...]}
    material_atual = '__default__'
    mtl_files = []

    for line in open(obj_path, 'r', encoding='utf-8', errors='ignore'):
        if line.startswith('#'):
            continue
        values = line.split()
        if not values:
            continue

        if values[0] == 'v':
            raw_vertices.append(values[1:4])
        elif values[0] == 'vt':
            raw_tex_coords.append(values[1:3])
        elif values[0] == 'vn':
            raw_normals.append(values[1:4])
            
        elif values[0] == 'mtllib':
            mtl_files.append(os.path.join(obj_dir, ' '.join(values[1:])))
        elif values[0] in ('usemtl', 'usemat'):
            material_atual = values[1]
            if material_atual not in faces_por_material:
                faces_por_material[material_atual] = []
        elif values[0] == 'f':
            face_v, face_uv, face_vn = [], [], []
            for token in values[1:]:
                parts = token.split('/')
                face_v.append(int(parts[0]))
                
                # Coordenadas de textura (Tratamento de strings vazias/ausentes)
                face_uv.append(int(parts[1]) if len(parts) >= 2 and parts[1] else 0)
                
                face_vn.append(int(parts[2]) if len(parts) >= 3 and parts[2] else 0)
                
            if material_atual not in faces_por_material:
                faces_por_material[material_atual] = []
            # Guardamos o conjunto completo associado ao material
            faces_por_material[material_atual].append((face_v, face_uv, face_vn))

    # ── 2. Parse do arquivo .mtl referenciados ─────────────────────────────
    materials = {}
    for mtl_path_ref in mtl_files:
        if os.path.exists(mtl_path_ref):
            materials.update(load_mtl(mtl_path_ref))
        else:
            print(f"[AVISO] MTL não encontrado: {mtl_path_ref}")

    if texture_overrides:
        for mat_name, tex_path in texture_overrides.items():
            if mat_name not in materials:
                materials[mat_name] = {'map_Kd': None, 'Kd': [1.0, 1.0, 1.0]}
            materials[mat_name]['map_Kd'] = tex_path

    # ── 3. Inserir vértices por grupo e carregar textura de cada material ─────
    grupos = []

    for mat_name, faces in faces_por_material.items():
        if not faces:
            continue

        vi_inicio = len(vertices_list)

        for face_v, face_uv, face_vn in faces:
            # Triangulação de posições
            for vid in circular_sliding_window_of_three(face_v):
                vertices_list.append(raw_vertices[vid - 1])
                
            # Triangulação de texturas
            for uid in circular_sliding_window_of_three(face_uv):
                if uid > 0:
                    u, v = raw_tex_coords[uid - 1]

                    u = float(u) * uv_mult_u
                    v = float(v) * uv_mult_v

                    textures_coord_list.append([u, v])
                else:
                    textures_coord_list.append([0.0, 0.0])
                    
            # Triangulação e alinhamento de normais 1:1 com os vértices
            for nid in circular_sliding_window_of_three(face_vn):
                if nid > 0:
                    normals_list.append(raw_normals[nid - 1])
                else:
                    # Fallback caso o modelo não possua normais mapeadas (Vetor apontando para cima)
                    normals_list.append(['0.0', '1.0', '0.0'])

        nv = len(vertices_list) - vi_inicio

        tex_path = (materials.get(mat_name) or {}).get('map_Kd')

        if tex_path and os.path.exists(tex_path):
            tid = load_texture_from_file(tex_path)
            print(f"  Material '{mat_name}': {nv} vértices → textura id={tid} ({os.path.basename(tex_path)})")
        else:
            tid = 0
            print(f"  [AVISO] Material '{mat_name}': sem textura, usando fallback (id=0)")

        grupos.append({'vertice_inicial': vi_inicio, 'num_vertices': nv, 'texture_id': tid})

    print(f"Modelo '{os.path.basename(obj_path)}' carregado: {len(grupos)} grupo(s) de material")
    return grupos

In [407]:
def load_cubemap(faces):
    # Carrega 6 imagens e as mapeia para as faces de um OpenGL Cubemap.
    # A ordem da lista 'faces' é: Right, Left, Top, Bottom, Front, Back.
    
    texture_id = glGenTextures(1)
    glBindTexture(GL_TEXTURE_CUBE_MAP, texture_id)

    for i in range(len(faces)):
        try:
            img = Image.open(faces[i])

            img = img.convert('RGBA') # Converte pra RGBA

            img_width = img.size[0]
            img_height = img.size[1]
            image_data = img.tobytes("raw", "RGBA", 0, 1)
            
            # POSITIVE_X é o primeiro target. Somar 'i' nos leva aos próximos targets automaticamente.
            glTexImage2D(GL_TEXTURE_CUBE_MAP_POSITIVE_X + i, 
                         0, GL_RGBA, img_width, img_height, 0, GL_RGBA, GL_UNSIGNED_BYTE, image_data)
        except Exception as e:
            print(f"Erro ao carregar textura da face {faces[i]}: {e}")

    # Configurações de filtragem
    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_MIN_FILTER, GL_LINEAR)
    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_MAG_FILTER, GL_LINEAR)
    
    # clamp to edge impede que apareçam linhas pretas nas quinas do cubo.
    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_S, GL_CLAMP_TO_EDGE)
    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_T, GL_CLAMP_TO_EDGE)
    glTexParameteri(GL_TEXTURE_CUBE_MAP, GL_TEXTURE_WRAP_R, GL_CLAMP_TO_EDGE)

    return texture_id

### Controlador de cena

In [408]:
# Carrega os dados da cena a partir de arquivos .json
def load_and_merge_scenes(filenames):
    merged_scene = []
    
    for filename in filenames:
        try:
            with open(filename, "r", encoding="utf-8") as f:
                scene_data = json.load(f)

                # Write the source file name in the objects
                for obj in scene_data:
                    obj["scene_file"] = filename

                # Add to the merged scene
                merged_scene.extend(scene_data)

        except FileNotFoundError:
            print(f"Error: File {filename} not found.")
        except json.JSONDecodeError:
            print(f"Error: Could not decode JSON in {filename}.")
            
    return merged_scene

def load_lights(filename):
    try:
        with open(filename, "r", encoding="utf-8") as f:
            lights = json.load(f)

    except FileNotFoundError:
        print(f"Error: File {filename} not found.")
    except json.JSONDecodeError:
        print(f"Error: Could not decode JSON in {filename}.")
    
    return lights

def save_lights(filename, lights):
    try:
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(lights, f, ensure_ascii=False, indent=4)

    except FileNotFoundError:
        print(f"Error: File {filename} not found.")

# Salva os dados da cena em seus respectivos arquivos .json
def save_scene(scene):
    # Coleta todos os nomes de arquivo únicos, usando 'scene.json' como fallback
    filenames = {obj.get('scene_file', 'scene.json') for obj in scene}

    for fn in filenames:
        clean_scene = [
            obj for obj in scene
            if obj.get('scene_file', 'scene.json') == fn # joga no scene.json por padrão se não existir arquivo no objeto 
        ]

        try:
            with open(fn, "w", encoding="utf-8") as f:
                json.dump(clean_scene, f, ensure_ascii=False, indent=4)

        except FileNotFoundError:
            print(f"Error: File {fn} not found.")


# Adiciona um objeto à cena
def add_to_scene(scene, obj_id, name, obj_file, texture_file=None, mtl_texture_overrides=None):
    """
    Adiciona um objeto à cena.

    Modelo com textura única:
        add_to_scene(scene, id, "nome", "modelo.obj", texture_file="tex.jpg")

    Modelo com múltiplos materiais (MTL sem map_Kd):
        add_to_scene(scene, id, "nome", "modelo.obj", mtl_texture_overrides={
            "material_1": "caminho/textura1.jpg",
            "material_2": "caminho/textura2.png",
        })
    """
    newObj = {
        "name": name,
        "obj_file": obj_file,
        "texture_file": texture_file,
        "mtl_texture_overrides": mtl_texture_overrides,
        # "angulo_obj": 0.0,
        "translacao": [0.0, 0.0, -20.0],
        "escala": [1.0, 1.0, 1.0],
        "rotacao": [0.0, 0.0, 0.0],
        # "id_textura": obj_id,
        # "obj_id": obj_id,
        "visible": True,
        "polygon": False,
        # "scene_id": len(scene)
        "illum_specs": {
            "ka": 0.2,
            "kd": 1.0,
            "ks": 0.2,
            "ns": 64.0
        },
        "ambient": "both",
        "uv_mult_u": 1.0,
        "uv_mult_v": 1.0
    }
    scene.append(newObj)
    return scene

def add_new_light(lights, name, pos=None, color=None, active=True, ambient=None):

    # Define valores padrão
    if pos is None:
        pos = [0.0,0.0,0.0]
    if color is None:
        color = [1.0,1.0,1.0]
    if ambient is None:
        ambient = "both"

    # Create new light
    new_light = {
        "name": name,
        "pos": pos,
        "color": color,
        "active": active,
        "ambient": ambient
    }

    lights.append(new_light)
    return lights

In [409]:
# Carrega a cena
scene_files = ["storage/scene.json"]
scene = load_and_merge_scenes(scene_files)

light_file = "storage/light.json"
lights = load_lights(light_file)

In [410]:
# add_to_scene(
#     scene,
#     obj_id=len(scene),      
#     name="Parede_Concreto", 
#     obj_file="objetos/wood-wall/WOOD_WALL.obj", 
#     mtl_texture_overrides={
#         "Material": "objetos/wood-wall/Plaster002_4K-JPG_Color.jpg",
#         "Material.001": "objetos/wood-wall/Plaster002_4K-JPG_Color.jpg"
#     }
# )
# save_scene(scene)

# add_new_light(
#     lights,
#     "internal_light_test",
#     pos=[0.0, 1.0, -22.0],
#     color=[0.0,1.0,1.0]
# )
# save_lights(light_file, lights)

In [411]:

# Old Day Skybox
# faces_skybox = [
#     "objetos/skybox/clouds1_east.bmp",
#     "objetos/skybox/clouds1_west.bmp",
#     "objetos/skybox/clouds1_up.bmp",
#     "objetos/skybox/clouds1_down.bmp",
#     "objetos/skybox/clouds1_north.bmp",
#     "objetos/skybox/clouds1_south.bmp"
# ]

# Skybox Noturna
faces_skybox = [
    "objetos/skybox/1.png",  # East (Right)
    "objetos/skybox/3.png",  # West (Left)
    "objetos/skybox/5.png",  # Up (Ceiling)
    "objetos/skybox/6.png",  # Down (Floor)
    "objetos/skybox/2.png",  # North (Front)
    "objetos/skybox/4.png"   # South (Back)
]

# Carrega o cubemap e salva o ID na variável
cubemap_texture_id = load_cubemap(faces_skybox)

### Carregamento dos Objetos da Cena

In [412]:
def carregar_objs():
    # Dicionario pra memorizar os objetos já carregados anteriormente
    cache_modelos = {} 

    triple_list = []

    for object in scene:
        # Chave do dicionário = arquivo_obj + mtl_overrides + arquivo_textura
        overrides_str = str(object.get('mtl_texture_overrides'))
        tex_str = str(object.get('texture_file'))
        cache_key = f"{object['obj_file']}_{overrides_str}_{tex_str}"

        # Verifica se esse modelo já foi lido antes
        if cache_key in cache_modelos:
            grupos = cache_modelos[cache_key]
            
        else:
            # Se não estiver, lê o arquivo
            if object.get('mtl_texture_overrides') is not None:
                print(f"Lendo (MLT_overrides): {object['obj_file']}")
                grupos = load_obj_with_mtl(object['obj_file'], object['mtl_texture_overrides'], object["uv_mult_u"], object["uv_mult_v"])
            else:
                print(f"Lendo (no_override): {object['obj_file']}")
                vi, nv, tid = load_obj_and_texture(object['obj_file'], [object['texture_file']])
                grupos = [{'vertice_inicial': vi, 'num_vertices': nv, 'texture_id': tid}]
                
            # Salva na cache
            cache_modelos[cache_key] = grupos

        triple_list.append(grupos)
    
    return triple_list

verticeInicial_quantosVertices_list = carregar_objs()


Lendo (MLT_overrides): objetos/bus_stop/bus_stop.obj
  Material 'blinn1SG': 1149120 vértices → textura id=2 (BusStopEnclosure.jpg)
  Material 'blinn2SG': 16632 vértices → textura id=3 (BusStopEnclosure.jpg)
Modelo 'bus_stop.obj' carregado: 2 grupo(s) de material
Lendo (MLT_overrides): objetos/onibus/bus_byjoao3DModels.obj
  Material 'base2': 211944 vértices → textura id=4 (base2_BaseColor.png)
  Material 'bus': 148782 vértices → textura id=5 (bus_BaseColor.png)
  Material 'glass': 3318 vértices → textura id=6 (glass_op.png)
Modelo 'bus_byjoao3DModels.obj' carregado: 3 grupo(s) de material
Lendo (MLT_overrides): objetos/oven/10122_Microwave_Oven_v1_L3.obj
  Material '10122_Microwave_Oven_v1_SG': 90864 vértices → textura id=7 (10122_Microwave_Oven_v1_Diffuse_SG.jpg)
Modelo '10122_Microwave_Oven_v1_L3.obj' carregado: 1 grupo(s) de material
Lendo (MLT_overrides): objetos/gnome/garden_gnome_4k.obj
  Material 'garden_gnome_01': 127854 vértices → textura id=8 (garden_gnome_diff_4k.jpg)
Modelo

### Envia Dados para a CPU

In [413]:
buffer_VBO = glGenBuffers(3) # Cria buffers

if(len(scene) != 0):
    # Vertices
    vertices = np.zeros(len(vertices_list), [("position", np.float32, 3)])
    vertices['position'] = vertices_list

    glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
    glBufferData(GL_ARRAY_BUFFER, vertices.nbytes, vertices, GL_STATIC_DRAW)
    stride = vertices.strides[0]
    offset = ctypes.c_void_p(0)
    loc_vertices = glGetAttribLocation(program, "position")
    glEnableVertexAttribArray(loc_vertices)
    glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, stride, offset)

    # Texturas
    textures = np.zeros(len(textures_coord_list), [("position", np.float32, 2)]) # duas coordenadas
    textures['position'] = textures_coord_list

    glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
    glBufferData(GL_ARRAY_BUFFER, textures.nbytes, textures, GL_STATIC_DRAW)
    stride = textures.strides[0]
    offset = ctypes.c_void_p(0)
    loc_texture_coord = glGetAttribLocation(program, "texture_coord")

    glEnableVertexAttribArray(loc_texture_coord)
    glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, stride, offset)

    # Normais
    normals = np.zeros(len(normals_list), [("position", np.float32, 3)]) # três coordenadas
    normals['position'] = normals_list


    # Upload coordenadas normals de cada vertice
    glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[2])
    glBufferData(GL_ARRAY_BUFFER, normals.nbytes, normals, GL_STATIC_DRAW)
    stride = normals.strides[0]
    offset = ctypes.c_void_p(0)
    loc_normals_coord = glGetAttribLocation(program, "normals")
    glEnableVertexAttribArray(loc_normals_coord)
    glVertexAttribPointer(loc_normals_coord, 3, GL_FLOAT, False, stride, offset)

### SkyBox

In [414]:
# Cubo pra SkyBox
skybox_vertices = np.array([
    # Positions (X, Y, Z)
    -1.0,  1.0, -1.0,
    -1.0, -1.0, -1.0,
     1.0, -1.0, -1.0,
     1.0, -1.0, -1.0,
     1.0,  1.0, -1.0,
    -1.0,  1.0, -1.0,

    -1.0, -1.0,  1.0,
    -1.0, -1.0, -1.0,
    -1.0,  1.0, -1.0,
    -1.0,  1.0, -1.0,
    -1.0,  1.0,  1.0,
    -1.0, -1.0,  1.0,

     1.0, -1.0, -1.0,
     1.0, -1.0,  1.0,
     1.0,  1.0,  1.0,
     1.0,  1.0,  1.0,
     1.0,  1.0, -1.0,
     1.0, -1.0, -1.0,

    -1.0, -1.0,  1.0,
    -1.0,  1.0,  1.0,
     1.0,  1.0,  1.0,
     1.0,  1.0,  1.0,
     1.0, -1.0,  1.0,
    -1.0, -1.0,  1.0,

    -1.0,  1.0, -1.0,
     1.0,  1.0, -1.0,
     1.0,  1.0,  1.0,
     1.0,  1.0,  1.0,
    -1.0,  1.0,  1.0,
    -1.0,  1.0, -1.0,

    -1.0, -1.0, -1.0,
    -1.0, -1.0,  1.0,
     1.0, -1.0, -1.0,
     1.0, -1.0, -1.0,
    -1.0, -1.0,  1.0,
     1.0, -1.0,  1.0
], dtype=np.float32)

# Buffer para a SkyBox
skybox_VBO = glGenBuffers(1)

# Upload dos vértices do skybox para a GPU
glBindBuffer(GL_ARRAY_BUFFER, skybox_VBO)
glBufferData(GL_ARRAY_BUFFER, skybox_vertices.nbytes, skybox_vertices, GL_STATIC_DRAW)

### Eventos

In [415]:
# Limites da Câmera
CAM_MAX_X = 20.0
CAM_MIN_X = -20.0
CAM_MAX_Y = 30.0
CAM_MIN_Y = -1.7
CAM_MAX_Z = 0.0
CAM_MIN_Z = -60.0

eixo = 1 # Eixo selecionado para rotação/escala
selected_obj = 0 # Objeto selecionado para a manipulação
uniform_scale = True # Toggle para escala uniforme
PolygonMode = False # Toggle para modo polígono
restrictMode = False # Toggle para modo restrito 
visible_light_objects = True # Toggle pros cubos das luzes
lightMode = True # Toggle pro controle das luzes
selected_light = 0 # Luz selecionada para a manipulação

# camera
cameraPos   = glm.vec3(0.0, 0.0, -50.0) # Posição Inicial da Câmera
cameraFront = glm.vec3(0.0, 0.0, 1.0) # Frente Inicial da Câmera
cameraUp    = glm.vec3(0.0, 1.0, 0.0) # Onde é cima

camera_base_speed = 40

fov   =  45.0

# timing
deltaTime = 0.0	# time between current frame and last frame
lastFrame = 0.0

firstMouse = True
yaw = 90.0 
pitch = 0.0
lastX =  largura/2
lastY =  altura/2

def key_event(window,key,scancode,action,mods):
    global eixo, selected_obj, scene, uniform_scale, PolygonMode, groups, restrictMode
    global cameraPos, cameraFront, cameraUp, camera_base_speed
    global scene_files, onibus_max_acc, onibus_spd, onibus_acc
    global light_bus_max_acc, light_bus_spd, light_bus_acc

    global light_bus, light_interior1, light_interior2
    global lights, selected_light, lightMode, visible_light_objects

    # Manipulações Gerais
    # Fechar janela
    if key == glfw.KEY_ESCAPE and action == glfw.PRESS:
        glfw.set_window_should_close(window, True)
    # Toggle Polygon Mode
    if key == glfw.KEY_P and action == glfw.PRESS: 
        PolygonMode = not PolygonMode
    # Reset
    if mods and glfw.MOD_CONTROL:
        if key == glfw.KEY_R and action == glfw.PRESS: 
            # Reseta a cena para o conteúdo dos arquivos .json
            scene = load_and_merge_scenes(scene_files)
            # Reseta parâmetros dos objetos
            onibus_acc = 0
            onibus_spd = 0
            light_bus_acc = 0
            light_bus_spd = 0
    
    if mods and glfw.MOD_SHIFT:
        camera_base_speed = 15
    else:
        camera_base_speed = 40
    
    if mods and glfw.MOD_SHIFT:
        if key == glfw.KEY_L and action == glfw.PRESS: 
            lightMode = not lightMode
    else:
        if key == glfw.KEY_L and action == glfw.PRESS: 
            visible_light_objects = not visible_light_objects

    # Restrict Mode
    if mods and (glfw.MOD_CONTROL & mods) and (glfw.MOD_SHIFT & mods) and (glfw.MOD_ALT & mods) and (glfw.MOD_SUPER & mods):
        if key == glfw.KEY_K and action == glfw.PRESS: 
            restrictMode = not restrictMode
    
    # Movimentação da câmera
    cameraSpeed = camera_base_speed * deltaTime
    if key == glfw.KEY_H and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += cameraSpeed * cameraFront
    
    if key == glfw.KEY_N and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= cameraSpeed * cameraFront
    
    if key == glfw.KEY_B and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos -= glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed
        
    if key == glfw.KEY_M and (action == glfw.PRESS or action == glfw.REPEAT):
        cameraPos += glm.normalize(glm.cross(cameraFront, cameraUp)) * cameraSpeed

    if restrictMode:
        cameraPos.x = max(CAM_MIN_X, min(CAM_MAX_X, cameraPos.x))
        cameraPos.y = max(CAM_MIN_Y, min(CAM_MAX_Y, cameraPos.y))
        cameraPos.z = max(CAM_MIN_Z, min(CAM_MAX_Z, cameraPos.z))

    # Manipulações de grupo exigidas pelo trabalho
    if restrictMode:
        
        # Aceleração [Onibus] (Translação) 
        if key == glfw.KEY_UP:
            if action == glfw.PRESS or action == glfw.REPEAT:
                onibus_acc = max(-onibus_max_acc, min(onibus_max_acc, onibus_acc - 0.001)) # Acelera pra frente
                light_bus_acc = onibus_acc
            elif action == glfw.RELEASE:
                onibus_acc = 0.0   # Para de acelerar quando solta a tecla
                light_bus_acc = 0.0
                
        if key == glfw.KEY_DOWN:
            if action == glfw.PRESS or action == glfw.REPEAT:
                onibus_acc = max(-onibus_max_acc, min(onibus_max_acc, onibus_acc + 0.001))  # Acelera pra trás
                light_bus_acc = onibus_acc
            elif action == glfw.RELEASE:
                onibus_acc = 0.0   # Para de acelerar quando solta a tecla
                light_bus_acc = 0.0







        if key == glfw.KEY_1 and (action == glfw.PRESS): # liga/desliga Luz Ônibus
            light_bus = not light_bus

        if key == glfw.KEY_2 and (action == glfw.PRESS): # liga/desliga Luz Interior 1
            light_interior1 = not light_interior1

        if key == glfw.KEY_3 and (action == glfw.PRESS): # liga/desliga Luz Interior 2
            light_interior2 = not light_interior2
        
    # Manipulações individuais utilizadas na montagem da cena
    else:

        if lightMode:
            # Muda Luz
            if key == glfw.KEY_Y and action == glfw.PRESS:
                selected_light = (selected_light+1) % len(lights)
            # Translação
            ## X
            if key == glfw.KEY_LEFT and (action == glfw.PRESS or action == glfw.REPEAT):
                lights[selected_light]['pos'][0] -= 0.01
            if key == glfw.KEY_RIGHT and (action == glfw.PRESS or action == glfw.REPEAT):
                lights[selected_light]['pos'][0] += 0.01
            ## Y
            if key == glfw.KEY_DOWN and (action == glfw.PRESS or action == glfw.REPEAT):
                lights[selected_light]['pos'][1] -= 0.01
            if key == glfw.KEY_UP and (action == glfw.PRESS or action == glfw.REPEAT):
                lights[selected_light]['pos'][1] += 0.01
            ## Z
            if key == glfw.KEY_Z and (action == glfw.PRESS or action == glfw.REPEAT):
                lights[selected_light]['pos'][2] -= 0.01
            if key == glfw.KEY_X and (action == glfw.PRESS or action == glfw.REPEAT):
                lights[selected_light]['pos'][2] += 0.01
                
        else:

            # Muda Objeto
            if key == glfw.KEY_Y and action == glfw.PRESS:
                selected_obj = (selected_obj+1) % len(scene)

            # Toggle Visibility
            if key == glfw.KEY_V and action == glfw.PRESS: 
                scene[selected_obj]['visible'] = not scene[selected_obj]['visible']

            # Toggle uniform_scale
            if key == glfw.KEY_C and action == glfw.PRESS:
                uniform_scale = not uniform_scale

            # Troca de Eixo Rotação
            if key == glfw.KEY_Q and action == glfw.PRESS:
                eixo = (eixo+1)%3

            # Rotação
            if key == glfw.KEY_A and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['rotacao'][eixo] += 1.0
            if key == glfw.KEY_D and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['rotacao'][eixo] -= 1.0

            # Escala
            if uniform_scale:
                if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
                    scene[selected_obj]['escala'][0] += scene[selected_obj]['escala'][0] * 0.05
                    scene[selected_obj]['escala'][1] += scene[selected_obj]['escala'][0] * 0.05
                    scene[selected_obj]['escala'][2] += scene[selected_obj]['escala'][0] * 0.05
                if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
                    scene[selected_obj]['escala'][0] -= scene[selected_obj]['escala'][0] * 0.05
                    scene[selected_obj]['escala'][1] -= scene[selected_obj]['escala'][0] * 0.05
                    scene[selected_obj]['escala'][2] -= scene[selected_obj]['escala'][0] * 0.05
            else:
                if key == glfw.KEY_W and (action == glfw.PRESS or action == glfw.REPEAT):
                    scene[selected_obj]['escala'][eixo] += 0.01
                if key == glfw.KEY_S and (action == glfw.PRESS or action == glfw.REPEAT):
                    scene[selected_obj]['escala'][eixo] -= 0.01

            # Translação
            ## X
            if key == glfw.KEY_LEFT and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['translacao'][0] -= 0.01
            if key == glfw.KEY_RIGHT and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['translacao'][0] += 0.01
            ## Y
            if key == glfw.KEY_DOWN and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['translacao'][1] -= 0.01
            if key == glfw.KEY_UP and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['translacao'][1] += 0.01
            ## Z
            if key == glfw.KEY_Z and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['translacao'][2] -= 0.01
            if key == glfw.KEY_X and (action == glfw.PRESS or action == glfw.REPEAT):
                scene[selected_obj]['translacao'][2] += 0.01
        

def framebuffer_size_callback(window, largura, altura):
    # make sure the viewport matches the new window dimensions note that width and 
    # height will be significantly larger than specified on retina displays.
    glViewport(0, 0, largura, altura)

# glfw: whenever the mouse moves, this callback is called
# -------------------------------------------------------
def mouse_callback(window, xpos, ypos):
    global cameraFront, lastX, lastY, firstMouse, yaw, pitch
   
    if (firstMouse):

        lastX = xpos
        lastY = ypos
        firstMouse = False

    xoffset = xpos - lastX
    yoffset = lastY - ypos # reversed since y-coordinates go from bottom to top
    lastX = xpos
    lastY = ypos

    sensitivity = 0.1 # change this value to your liking
    xoffset *= sensitivity
    yoffset *= sensitivity

    yaw += xoffset
    pitch += yoffset

    # make sure that when pitch is out of bounds, screen doesn't get flipped
    if (pitch > 89.0):
        pitch = 89.0
    if (pitch < -89.0):
        pitch = -89.0

    front = glm.vec3()
    front.x = glm.cos(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    front.y = glm.sin(glm.radians(pitch))
    front.z = glm.sin(glm.radians(yaw)) * glm.cos(glm.radians(pitch))
    cameraFront = glm.normalize(front)

# glfw: whenever the mouse scroll wheel scrolls, this callback is called
# ----------------------------------------------------------------------
def scroll_callback(window, xoffset, yoffset):
    global fov

    fov -= yoffset
    if (fov < 1.0):
        fov = 1.0
    if (fov > 45.0):
        fov = 45.0
    
glfw.set_key_callback(window,key_event)
glfw.set_framebuffer_size_callback(window, framebuffer_size_callback)
glfw.set_cursor_pos_callback(window, mouse_callback)
glfw.set_scroll_callback(window, scroll_callback)

# tell GLFW to capture our mouse
glfw.set_input_mode(window, glfw.CURSOR, glfw.CURSOR_DISABLED)

### Matrizes Model, View e Projection

In [416]:
def model(r_x, r_y, r_z, t_x, t_y, t_z, s_x, s_y, s_z):
    # r_x, r_y, r_z são ângulos de Euler (em graus) em torno de X, Y, Z.

    matrix_transform = glm.mat4(1.0) # instanciando uma matriz identidade

    # Ordem aplicada aos vértices: Scale -> Rotate -> Translate
    # (em GLM/coluna-major, basta multiplicar nesta ordem: T * R * S)

    # aplicando translacao
    matrix_transform = glm.translate(matrix_transform, glm.vec3(t_x, t_y, t_z))

    # aplicando rotacao em cada eixo (ângulos de Euler X, Y, Z)
    matrix_transform = glm.rotate(matrix_transform, math.radians(r_x), glm.vec3(1.0, 0.0, 0.0))
    matrix_transform = glm.rotate(matrix_transform, math.radians(r_y), glm.vec3(0.0, 1.0, 0.0))
    matrix_transform = glm.rotate(matrix_transform, math.radians(r_z), glm.vec3(0.0, 0.0, 1.0))

    # aplicando escala
    matrix_transform = glm.scale(matrix_transform, glm.vec3(s_x, s_y, s_z))

    matrix_transform = np.array(matrix_transform)

    return matrix_transform

def view():
    global cameraPos, cameraFront, cameraUp
    mat_view = glm.lookAt(cameraPos, cameraPos + cameraFront, cameraUp);
    mat_view = np.array(mat_view)
    return mat_view

def projection():
    global altura, largura
    # perspective parameters: fovy, aspect, near, far
    mat_projection = glm.perspective(glm.radians(fov), largura/altura, 0.1, 100000.0)

    
    mat_projection = np.array(mat_projection)    
    return mat_projection

## Desenha Objetos

In [417]:
def desenha_obj(rot_coords, transl_coords, escal_coords, grupos, illum_specs):
    """
    Desenha um objeto 3D aplicando a matriz model calculada a partir das transformações.

    grupos: lista de grupos de renderização, cada um com:
            {'vertice_inicial': int, 'num_vertices': int, 'texture_id': int}
    Modelos com uma única textura têm lista de 1 elemento.
    Modelos com múltiplos materiais têm um elemento por material.
    """
    mat_model = model(*rot_coords, *transl_coords, *escal_coords)
    loc_model = glGetUniformLocation(program, "model")
    glUniformMatrix4fv(loc_model, 1, GL_TRUE, mat_model)

    loc_ka = glGetUniformLocation(program, "ka") # recuperando localizacao da variavel ka na GPU
    glUniform1f(loc_ka, illum_specs["ka"]) ### envia ka pra gpu
    
    loc_kd = glGetUniformLocation(program, "kd") # recuperando localizacao da variavel kd na GPU
    glUniform1f(loc_kd, illum_specs["kd"]) ### envia kd pra gpu    
    
    loc_ks = glGetUniformLocation(program, "ks") # recuperando localizacao da variavel ks na GPU
    glUniform1f(loc_ks, illum_specs["ks"]) ### envia ks pra gpu        
    
    loc_ns = glGetUniformLocation(program, "ns") # recuperando localizacao da variavel ns na GPU
    glUniform1f(loc_ns, illum_specs["ns"]) ### envia ns pra gpu   

    for grupo in grupos:
        glBindTexture(GL_TEXTURE_2D, grupo['texture_id'])
        glDrawArrays(GL_TRIANGLES, grupo['vertice_inicial'], grupo['num_vertices'])

### Exibição e Loop Principal de Janela


In [418]:
glfw.show_window(window)

In [419]:
# Atributos Onibus
onibus_acc = 0 # Aceleração do Onibus
onibus_spd = 0 # Velocidade do Onibus
onibus_max_acc = 0.005 # Aceleração máxima do onibus
onibus_max_speed = 0.5 # Velocidade máxima do onibus
onibus_atrito = 0.98

# Atributos light_bus
light_bus_acc = 0 # Aceleração do light_bus
light_bus_spd = 0 # Velocidade do light_bus
light_bus_max_acc = 0.005 # Aceleração máxima do light_bus
light_bus_max_speed = 0.5 # Velocidade máxima do light_bus
light_bus_atrito = 0.98

# Luzes
light_bus = False
light_interior1 = False
light_interior2 = False

# Procura os objetos na cena
targets = ["Onibus", "light_bus", "clock", "Cubo"]
targets_indexes = [None] * len(targets)

for i in range(len(targets)):
    for j, obj in enumerate(scene):
        if obj.get('name') == targets[i]:
            targets_indexes[i] = j
            break

def special_object_transform():
    global scene, targets_indexes, deltaTime
    global light_bus, light_interior1, light_interior2
    global onibus_acc, onibus_spd, onibus_max_acc, onibus_max_speed, onibus_atrito
    global light_bus_acc, light_bus_spd, light_bus_max_acc, light_bus_max_speed, light_bus_atrito

    # Visibilidade do Onibus
    if targets_indexes[0] is not None:
        if scene[targets_indexes[0]]['translacao'][0] >= 270 or scene[targets_indexes[0]]['translacao'][0] <= -270:
            scene[targets_indexes[0]]['visible'] = False
        else:
            scene[targets_indexes[0]]['visible'] = True

    # Translação (Onibus)

    # Aumenta a velocidade de acordo com a aceleração
    # exceto quando alcança o limite de velocidade
    onibus_spd = max(-onibus_max_speed, min(onibus_max_speed, onibus_spd + onibus_acc * deltaTime))
    light_bus_spd = onibus_spd

    # Se aceleração = 0, começa a desacelar por atrito
    if onibus_acc == 0:
        # Zera speed (safeguard)
        if onibus_spd < 0.0005 and onibus_spd > -0.0005:
            onibus_spd = 0
            light_bus_spd = 0
        else:
            onibus_spd *= onibus_atrito
            light_bus_spd = onibus_spd

    
    # Desloca o onibus e a light_bus de acordo com a velocidade
    if targets_indexes[0] is not None:
        scene[targets_indexes[0]]['translacao'][0] += onibus_spd * deltaTime
        # scene[targets_indexes[1]]['translacao'][0] += onibus_spd * deltaTime

def object_shader_config(mat_view, mat_projection):
    global mainShader, program, cameraPos, buffer_VBO, vertices, textures, normals

    mainShader.use() # Ativa o shader normal

    # Salva posição da câmera na GPU
    loc_view_pos = glGetUniformLocation(program, "viewPos") # recuperando localizacao da variavel viewPos na GPU (frag shader)
    glUniform3f(loc_view_pos, cameraPos[0], cameraPos[1], cameraPos[2]) ### posicao da camera/observador (x,y,z)

    # Define cor ambiente
    loc_color_tint = glGetUniformLocation(program, "colorTint")
    glUniform3f(loc_color_tint, 1.0, 1.0, 1.0)
    
    # Reconecta os buffers normais
    glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[0])
    loc_vertices = glGetAttribLocation(program, "position")
    glEnableVertexAttribArray(loc_vertices)
    glVertexAttribPointer(loc_vertices, 3, GL_FLOAT, False, vertices.strides[0], ctypes.c_void_p(0))

    glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[1])
    loc_texture_coord = glGetAttribLocation(program, "texture_coord")
    glEnableVertexAttribArray(loc_texture_coord)
    glVertexAttribPointer(loc_texture_coord, 2, GL_FLOAT, False, textures.strides[0], ctypes.c_void_p(0))

    glBindBuffer(GL_ARRAY_BUFFER, buffer_VBO[2])
    loc_normals_coord = glGetAttribLocation(program, "normals")
    glEnableVertexAttribArray(loc_normals_coord)
    glVertexAttribPointer(loc_normals_coord, 3, GL_FLOAT, False, normals.strides[0], ctypes.c_void_p(0))
    
    # Envia as matrizes para o shader normal
    loc_view = glGetUniformLocation(program, "view")
    glUniformMatrix4fv(loc_view, 1, GL_TRUE, mat_view)
    loc_projection = glGetUniformLocation(program, "projection")
    glUniformMatrix4fv(loc_projection, 1, GL_TRUE, mat_projection)

def light_gpu_config(ambient):
    global lights, program

    MAX_LIGHTS = 10 # Número max de luzes

    light_pos = []
    light_colors = []

    num_lights = 0
    for light in lights:
        if light["active"]:
            if (light["ambient"] == ambient) or (light["ambient"] == "both") or ambient == "both":
                num_lights += 1
                light_pos.extend(light["pos"])
                light_colors.extend(light["color"])
        
        if num_lights >= MAX_LIGHTS:
            break

    np_positions = np.array(light_pos, dtype=np.float32)
    np_colors = np.array(light_colors, dtype=np.float32)

    # Pass the number of lights
    num_lights_loc = glGetUniformLocation(program, "numLights")
    glUniform1i(num_lights_loc, num_lights)

    # Get the memory locations for the arrays
    pos_array_loc = glGetUniformLocation(program, "lightPositions[0]")
    col_array_loc = glGetUniformLocation(program, "lightColors[0]")

    if num_lights > 0:
        glUniform3fv(pos_array_loc, num_lights, np_positions)
        glUniform3fv(col_array_loc, num_lights, np_colors)

    # print(ambient)
    # print(lights)
    # print(light_pos)
    # print(np_positions)
    # print(num_lights)
    

def skybox_shader_config(mat_projection):
    global skyboxShader, skybox_VBO, skyboxProgram, cubemap_texture_id

    # SkyBox
    glDepthFunc(GL_LEQUAL)  # Muda a função de profundidade
    skyboxShader.use()      # Ativa o shader do skybox
    
    # Remove a translação da matriz de View
    mat_view_skybox = glm.mat4(glm.mat3(view())) 
    mat_view_skybox = np.array(mat_view_skybox)
    
    # Envia as matrizes para o shader do skybox
    loc_view_sky = glGetUniformLocation(skyboxProgram, "view")
    glUniformMatrix4fv(loc_view_sky, 1, GL_TRUE, mat_view_skybox)
    loc_proj_sky = glGetUniformLocation(skyboxProgram, "projection")
    glUniformMatrix4fv(loc_proj_sky, 1, GL_TRUE, mat_projection)

    # Conecta o buffer exclusivo do Skybox
    glBindBuffer(GL_ARRAY_BUFFER, skybox_VBO)
    loc_sky_pos = glGetAttribLocation(skyboxProgram, "position")
    glEnableVertexAttribArray(loc_sky_pos)
    # Stride é 3 * 4 bytes (12 bytes por vértice, XYZ)
    glVertexAttribPointer(loc_sky_pos, 3, GL_FLOAT, False, 12, ctypes.c_void_p(0))
    
    # Ativa a textura Cubemap 
    glActiveTexture(GL_TEXTURE0)
    glBindTexture(GL_TEXTURE_CUBE_MAP, cubemap_texture_id)
    skyboxShader.setInt("imagem_cube", 0)

    # Desenha os 36 vértices do cubo
    glDrawArrays(GL_TRIANGLES, 0, 36)

def desenha_luzes():
    global lights, targets_indexes, scene

    cubo = scene[targets_indexes[3]]
    grupos  = verticeInicial_quantosVertices_list[targets_indexes[3]]

    for light in lights:
        loc_color_amb = glGetUniformLocation(program, "colorAmbience")
        glUniform3f(loc_color_amb, light["color"][0], light["color"][1], light["color"][2])

        desenha_obj([0.0, 0.0, 0.0], light["pos"], cubo['escala'], grupos, cubo["illum_specs"])
    

In [420]:
glEnable(GL_DEPTH_TEST)

# Loop principal, continua enquanto a janela estiver aberta
while not glfw.window_should_close(window):    
    # Calcula Delta Time
    currentFrame = glfw.get_time()
    deltaTime = currentFrame - lastFrame
    lastFrame = currentFrame

    glfw.poll_events() 
       
    glClear(GL_COLOR_BUFFER_BIT | GL_DEPTH_BUFFER_BIT)
    glClearColor(1.0, 1.0, 1.0, 1.0)

    # Ajusta nome da janela de acordo com o modo restrito
    if restrictMode:
        glfw.set_window_title(window, "Casa no Meio do Nada")
    else:
        if lightMode:
            glfw.set_window_title(window, f"name: {lights[selected_light]['name']}, selected_light: {selected_light}, pos: {lights[selected_light]["pos"]}")
        else:
            glfw.set_window_title(window, f"name: {scene[selected_obj]['name']}, selected_obj: {selected_obj}, uniform_scale: {uniform_scale}, eixo: {eixo}")

    # View e Projection calculados uma vez por frame
    mat_view = view()
    mat_projection = projection()

    # Configura Shader de Objeto
    object_shader_config(mat_view, mat_projection)

    # Aplica transformações a objetos controláveis
    special_object_transform()

    # Tipos de Ambiente
    ambient_types = ["internal", "external", "both"]

    # Loop
    for ambient_mode in ambient_types:
        light_gpu_config(ambient_mode) # Configura apenas as luzes desse ambiente
        
        # Renderiza apenas os objetos desse ambiente
        for i, obj in enumerate(scene):
            if obj["ambient"] == ambient_mode:
                glPolygonMode(GL_FRONT_AND_BACK, GL_LINE if PolygonMode else GL_FILL)
                
                if not obj['visible']:
                    continue

                grupos = verticeInicial_quantosVertices_list[i]
                desenha_obj(obj['rotacao'], obj['translacao'], obj['escala'], grupos, obj["illum_specs"])
    
    # Desenha os objetos das luzes
    if visible_light_objects:
        desenha_luzes()
    
    # Configura e aplica shader skybox 
    skybox_shader_config(mat_projection)
    
    glDepthFunc(GL_LESS) # Retorna a função de profundidade ao normal para o próximo frame

    glfw.swap_buffers(window)
    # break


glfw.terminate()

if not restrictMode:
    save_scene(scene) # Salva a cena
    save_lights(light_file, lights)